# 小红书爆款文案生成 — 本地 Ollama 版

基于本地 Ollama `deepseek-r1:8b` 模型，使用 `requests` 直接调用 API。
不依赖 OpenAI SDK、Tavily、Milvus 等外部工具。

## 1. 配置

In [1]:
import json
import re
import logging
import requests

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    filename="./rednote-agent.log",
    filemode="a",
    encoding="utf-8",
    force=True,
)
logger = logging.getLogger("rednote-agent")

OLLAMA_URL = "http://localhost:11434/v1/chat/completions"
DEEPSEEK_MODEL = "deepseek-r1:8b"

logger.info("配置初始化完成")

## 2. LLM 调用与 JSON 提取

In [2]:
def call_ollama(messages: list[dict], temperature: float = 0.7, timeout: int = 300) -> str:
    """直接通过 requests 调用 Ollama API，返回模型响应文本。"""
    payload = {
        "model": DEEPSEEK_MODEL,
        "messages": messages,
        "temperature": temperature,
    }
    resp = requests.post(OLLAMA_URL, json=payload, timeout=timeout)
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]


def extract_json_from_r1(content: str) -> str | None:
    """从 R1 模型响应中提取 JSON，处理 <think> 标签。"""
    cleaned = re.sub(r'<think>.*?</think>', '', content, flags=re.DOTALL).strip()

    # 匹配 ```json { ... } ```
    match = re.search(r'```json\s*(\{.*?\})\s*```', cleaned, re.DOTALL)
    if match:
        try:
            return json.dumps(json.loads(match.group(1)), ensure_ascii=False, indent=2)
        except json.JSONDecodeError:
            pass

    # 直接找含 title+body 的 JSON 对象
    match = re.search(r'(\{[^{}]*"title"[^{}]*"body"[^{}]*\})', cleaned, re.DOTALL)
    if match:
        try:
            return json.dumps(json.loads(match.group(1)), ensure_ascii=False, indent=2)
        except json.JSONDecodeError:
            pass

    # 整段解析
    try:
        return json.dumps(json.loads(cleaned), ensure_ascii=False, indent=2)
    except json.JSONDecodeError:
        return None

## 3. 文案生成引擎

In [3]:
SYSTEM_PROMPT = (
    "你是一个资深的小红书爆款文案专家，擅长结合最新潮流和产品卖点，"
    "创作引人入胜、高互动、高转化的笔记文案。\n\n"
    "请直接以 JSON 格式输出最终文案，用 ```json ... ``` 包裹，格式如下：\n"
    "```json\n"
    "{\n"
    '  "title": "小红书标题",\n'
    '  "body": "小红书正文",\n'
    '  "hashtags": ["#标签1", "#标签2", "#标签3", "#标签4", "#标签5"],\n'
    '  "emojis": ["✨", "🔥", "💖", "💯", "🎉"]\n'
    "}\n"
    "```\n"
)


def generate_rednote(product_name: str, tone_style: str = "活泼甜美", max_retries: int = 3) -> str:
    """生成小红书爆款文案。"""
    logger.info(f"🚀 启动文案生成，产品：{product_name}，风格：{tone_style}")

    user_prompt = (
        f"请为产品「{product_name}」生成一篇小红书爆款文案。\n"
        f"要求：语气{tone_style}，包含标题、正文、至少5个标签和5个表情符号。\n\n"
        f"请直接输出 JSON，用 ```json ... ``` 包裹。"
    )
    logger.info(f"用户提示词：{user_prompt}")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    for attempt in range(1, max_retries + 1):
        logger.info(f"-- 生成尝试 {attempt}/{max_retries} --")
        try:
            content = call_ollama(messages)
            logger.info(f"[LLM 响应] {content[:300]}...")

            result = extract_json_from_r1(content)
            if result:
                logger.info("✅ 文案生成成功")
                return result

            logger.warning(f"尝试 {attempt}: 未能提取有效 JSON")
        except Exception as e:
            logger.error(f"尝试 {attempt} 失败: {e}")

    logger.warning("⚠️ 所有尝试均失败")
    return json.dumps(
        {"title": "生成失败", "body": "未能生成文案，请检查 Ollama 服务。", "hashtags": [], "emojis": []},
        ensure_ascii=False, indent=2,
    )

## 4. 输出格式化

In [4]:
def format_rednote_for_markdown(json_string: str) -> str:
    """将 JSON 文案转换为 Markdown 格式。"""
    try:
        data = json.loads(json_string)
    except json.JSONDecodeError as e:
        return f"错误：无法解析 JSON - {e}\n原始：\n{json_string}"

    title = data.get("title", "无标题")
    body = data.get("body", "")
    hashtags = data.get("hashtags", [])
    emojis = data.get("emojis", [])

    md = f"## {title}\n\n{body}\n"
    if emojis:
        md += f"\n{''.join(emojis)}\n"
    if hashtags:
        md += f"\n{' '.join(hashtags)}\n"
    return md.strip()

## 5. 端到端测试

In [7]:
result = generate_rednote("蓝牙降噪耳机", "严肃科普")

print("--- 格式化后的小红书文案 (Markdown) ---")
print(format_rednote_for_markdown(result))

--- 格式化后的小红书文案 (Markdown) ---
## 蓝牙降噪耳机：声音科技的革命性突破

蓝牙降噪耳机已成为现代音频设备的标杆，其核心卖点在于主动降噪（ANC）技术，通过算法实时捕捉并抵消环境噪音，提供高达30dB的降噪效果。这项技术基于物理声波原理，利用麦克风捕捉外部声音，然后通过扬声器反向产生抵消波，实现沉浸式音频体验。蓝牙5.0标准的采用进一步提升了连接稳定性，减少断连问题，同时支持AAC音频编码，确保高保真音质。数据显示，这类耳机的平均续航可达20小时，适合日常通勤或工作场景。更重要的是，研究显示，降噪耳机能减少环境噪音干扰，提升专注力，对心理健康有积极影响。如果你正在寻找一款高效、可靠的音频伴侣，这款蓝牙降噪耳机绝对是值得投资的选择。欢迎在评论区分享你的使用心得，或讨论如何提升音频体验！

✨🔥💖💯🎉

#蓝牙耳机 #降噪技术 #Audio科技 #无线音频 #耳机推荐


## 6. 导出文案为 Markdown 文件

In [8]:
from datetime import datetime
from pathlib import Path


def export_rednote_to_md(json_string: str, output_dir: str = "md") -> str:
    """导出文案为 Markdown 文件。"""
    md_content = format_rednote_for_markdown(json_string)
    try:
        title = json.loads(json_string).get("title", "未命名文案")
    except Exception:
        title = "未命名文案"

    safe_title = re.sub(r'[\\/:*?"<>|]', '_', title)[:50]
    filename = f"{safe_title}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md"

    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    file_path = out_path / filename
    file_path.write_text(md_content, encoding="utf-8")
    logger.info(f"[Export] 文案已导出: {file_path}")
    print(f"✅ 文案已导出到: {file_path}")
    return str(file_path)


export_rednote_to_md(result)

✅ 文案已导出到: md\蓝牙降噪耳机：声音科技的革命性突破_20260410_165621.md


'md\\蓝牙降噪耳机：声音科技的革命性突破_20260410_165621.md'